### casual Self Attention
Wq,Wv,Wk are the trainable weights 
mask the token after the query 
only before the query token participate into the attention score 

In [445]:
import torch
inputs=torch.tensor([[0.22,0.44,0.56], # your 
                    [0.33,-0.144,0.756], # journey
                    [0.22,0.344,0.856], # starts
                    [-0.722,-0.644,-0.356], # with
                    [0.022,0.744,0.1356], # one
                    [0.122,0.044,0.956]]) # step 

In [446]:
d_in=inputs[0].shape[0]
d_out=2
print(d_in)

3


In [447]:
wk=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
wq=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
wv=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)

In [448]:
print(wk)

Parameter containing:
tensor([[0.8420, 0.7596],
        [0.1454, 0.2150],
        [0.7413, 0.7641]])


In [449]:
## make query,key,values matrix
query=inputs@wq
key=inputs@wk
value=inputs@wv

In [450]:
print(key)

tensor([[ 0.6643,  0.6896],
        [ 0.8174,  0.7973],
        [ 0.8698,  0.8951],
        [-0.9655, -0.9589],
        [ 0.2272,  0.2802],
        [ 0.8178,  0.8326]])


In [451]:
## make attention socre 
attention_score=query@key.T
attention_score.shape

torch.Size([6, 6])

In [452]:
attention_score[1] ## attention score of journet with all other keys

tensor([ 0.1712,  0.2085,  0.2238, -0.2470,  0.0603,  0.2101])

In [453]:
d_key=key.shape[-1]
d_key

2

In [454]:
attention_wt= torch.softmax(attention_score/d_key**0.5 ,dim=-1)
attention_wt

tensor([[0.1845, 0.2032, 0.2137, 0.0573, 0.1361, 0.2053],
        [0.1736, 0.1782, 0.1802, 0.1291, 0.1605, 0.1784],
        [0.1841, 0.2020, 0.2117, 0.0606, 0.1378, 0.2038],
        [0.0868, 0.0744, 0.0688, 0.5559, 0.1408, 0.0733],
        [0.1855, 0.2077, 0.2208, 0.0462, 0.1293, 0.2105],
        [0.1779, 0.1863, 0.1904, 0.1039, 0.1545, 0.1870]])

In [455]:
## we want only before the word attention score and wt and contributions


### Mask the attention score before the query

In [456]:
context_length=attention_score.shape[0] ## it give how many words go inside the 
mask_sample=torch.tril(torch.ones(context_length,context_length))
mask_sample

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

### Data leak in this method
after the find the attention wt we mask them 
due to which the contribution of the after token by the query is exist 

In [457]:
torch.sum(attention_wt[1])

tensor(1.0000)

In [458]:
masked_attention_wt=attention_wt*mask_sample
torch.sum(masked_attention_wt[1])

tensor(0.3518)

In [459]:
masked_attention_wt[1]

tensor([0.1736, 0.1782, 0.0000, 0.0000, 0.0000, 0.0000])

In [460]:
## to handle this we normalized it 
row_sum=masked_attention_wt.sum(dim=1,keepdim=True)
row_sum

tensor([[0.1845],
        [0.3518],
        [0.5978],
        [0.7859],
        [0.7895],
        [1.0000]])

In [461]:
masked_attention_wt=masked_attention_wt/row_sum
masked_attention_wt

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4934, 0.5066, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3080, 0.3379, 0.3541, 0.0000, 0.0000, 0.0000],
        [0.1105, 0.0947, 0.0875, 0.7074, 0.0000, 0.0000],
        [0.2349, 0.2631, 0.2796, 0.0586, 0.1638, 0.0000],
        [0.1779, 0.1863, 0.1904, 0.1039, 0.1545, 0.1870]])

### After attention socre Mask them 

In [462]:
mask=torch.triu(torch.ones(context_length,context_length),diagonal=1)
### diagonal =1 bec it make the diagonal element 0
## it give that diagonal = true means make the diagonal =0
mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [463]:
mask_inf_attention_score=attention_score.masked_fill(mask.bool(),-torch.inf)
mask_inf_attention_score

tensor([[ 0.6811,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.1712,  0.2085,    -inf,    -inf,    -inf,    -inf],
        [ 0.6469,  0.7777,  0.8442,    -inf,    -inf,    -inf],
        [-1.0812, -1.2990, -1.4109,  1.5448,    -inf,    -inf],
        [ 0.8100,  0.9701,  1.0565, -1.1547,  0.3000,    -inf],
        [ 0.3128,  0.3775,  0.4084, -0.4483,  0.1134,  0.3827]])

In [464]:
attention_wt=torch.softmax(mask_inf_attention_score/d_key**0.5,dim=-1)
attention_wt

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4934, 0.5066, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3080, 0.3379, 0.3541, 0.0000, 0.0000, 0.0000],
        [0.1105, 0.0947, 0.0875, 0.7074, 0.0000, 0.0000],
        [0.2349, 0.2631, 0.2796, 0.0586, 0.1638, 0.0000],
        [0.1779, 0.1863, 0.1904, 0.1039, 0.1545, 0.1870]])

### Dropout introduce 
we use dropout rate = 50 %


In [470]:
torch.manual_seed(123)
dropout=torch.nn.Dropout(0.5)
ex=torch.ones(6,6)
dropout(ex)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [471]:
dropout(attention_wt) ## introduce the dropout inside the attention weights

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6757, 0.7083, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.1750, 1.4147, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3276, 0.0000],
        [0.3559, 0.3725, 0.0000, 0.2078, 0.3091, 0.3739]])